In [11]:
"""
find_tif_files.py

Searches for .tif files under datadir matching the reporter/condition table.
File IDs like "0822-02" are matched exactly: xy02 will NOT match xy12/xy22.

Usage:
    python find_tif_files.py

Output:
    - Prints a summary table to stdout
    - Saves results to tif_file_mapping.csv
"""

import os
import re
import csv
from pathlib import Path

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATADIR = Path("${SPARK_ACQUISITION_ROOT}/ExRai-imaging")

SUBFOLDERS = [
    "20240321-ECL_CY_HT1080_ExRai_polydIdC_2ug_ml_trials002",
    "20240321-ECL_CY_HT1080_ExRai_polyIC_2ug_ml_trials002",
    "20240404-CY_HT1080_ExRai_1uM_diABZI_",
    "20240404-_CY_HT1080_ExRai_polyIC_7ug_ml",
    "20240516-_CY_HT1080_ExRai_abiZI_001",
    "20240516-_cy_ht1080_exrai_polyic",
    "20240516-_CY_HT1080_ExRai_polyIC",
    "20240606-_CY_HT1080_ExRai_diABZI001",
    "20240606-_CY_HT1080_ExRai_polyIC002",
    "20240627-plate1-polyIC_600",
    "20240627-plate2-polyIC_600",
    "20240628-plate1-polyIC_600",
    "20240628-plate2-polyIC_600",
    "20240718-CY_spk_diabzi_1uM_2uM",
    "20240719-CY_A3_fast",
    "20240719-CY_A3_fast002",
    "20240719-CY_all",
    "20240725_F5", "20240725_F6", "20240725_F7",
    "20240725_F8", "20240725_F9",
    "20240725_r1", "20240725_r2",
    "20240814_cy_plate1", "20240814_cy_plate2",
    "20240815_cy_plate3", "20240815_cy_plate4",
    "20240822_cy_plate1", "20240822_cy_plate2",
    "20250126-plate1", "20250130-plate1",
    "240731_plate1", "240731_plate1_final",
    "240731_plate2", "240731_plate3",
    "240801_plate4",
]

# ── TABLE (reporter, condition) -> [raw file IDs] ────────────────────────────
RAW_TABLE = {
    ("tbk1",       "blank"):      ["240731p2-50"],
    ("tbk1",       "polyIC"):     ["0731p2-57", "0725r1xy18"],
    ("tbk1",       "diABZI"):     ["0731p2-62"],
    ("tbk1",       "BMS-345541"): ["0725r1xy10"],
    ("tbk1-novel", "blank"):      ["240731p3-34"],
    ("tbk1-novel", "polyIC"):     ["0731p3-60", "0731p3-40"],
    ("tbk1-novel", "diABZI"):     ["0731p3-46", "0731p3-65"],
    ("tbk1-novel", "BMS-345541"): ["0731p3-36"],
    ("ikkb",       "blank"):      ["0822-02"],
    ("ikkb",       "TNFa"):       ["0822-13"],
    ("ikkb",       "diABZI"):     ["0731p2-46"],
    ("ikkb",       "BMS-345541"): ["240822-08"],
    ("ikkb-novel", "blank"):      ["0822-18", "0822-20"],
    ("ikkb-novel", "TNFa"):       ["0822-28"],
    ("ikkb-novel", "diABZI"):     ["0731p3-12", "0731p3-30"],
    ("ikkb-novel", "BMS-345541"): ["240822-26"],
    ("ikkb-novel", "blank_alt"):  ["240731_plate3c1xy03"],
}


# ── HELPERS ──────────────────────────────────────────────────────────────────

def normalize_date_prefix(s: str) -> str:
    """Expand short date prefixes: 0731 -> 240731, 0822 -> 240822."""
    s = s.strip()
    if re.match(r'^0[0-9]{3}', s):
        s = '24' + s
    return s


def parse_file_id(raw_id: str) -> dict:
    """
    Parse a raw file ID into date, plate, xy_raw, filename_prefix.

    xy_raw keeps the original digit string (e.g. "02") so the matcher
    can do an exact boundary match and not confuse xy02 with xy12.

    Examples:
      "0731p2-57"           -> date=240731, plate=p2, xy_raw="57"
      "0725r1xy18"          -> date=240725, plate=r1, xy_raw="18"
      "0822-02"             -> date=240822, plate=p1, xy_raw="02"
      "240822-08"           -> date=240822, plate=p1, xy_raw="08"
      "240731_plate3c1xy03" -> date=240731, plate=p3, prefix=c1, xy_raw="03"
    """
    info = {"raw": raw_id, "date": None, "plate": None,
            "xy_raw": None, "filename_prefix": None}
    s = normalize_date_prefix(raw_id.strip())

    # 240731_plate3c1xy03  ->  plate3, prefix c1, xy 03
    m = re.match(r'(\d{6})_plate(\d+)(c\d+)xy(\d+)', s, re.IGNORECASE)
    if m:
        info["date"] = m.group(1)
        info["plate"] = "p" + m.group(2)
        info["filename_prefix"] = m.group(3).lower()
        info["xy_raw"] = m.group(4)
        return info

    # 240731p2-57  /  0731p3-60
    m = re.match(r'(\d{6})(p\d+|r\d+)-(\d+)', s, re.IGNORECASE)
    if m:
        info["date"] = m.group(1)
        info["plate"] = m.group(2).lower()
        info["xy_raw"] = m.group(3)
        return info

    # 240725r1xy18  /  240731p3xy46
    m = re.match(r'(\d{6})(r\d+|p\d+)xy(\d+)', s, re.IGNORECASE)
    if m:
        info["date"] = m.group(1)
        info["plate"] = m.group(2).lower()
        info["xy_raw"] = m.group(3)
        return info

    # 0822-02  /  240822-08  (all 0822 are plate1)
    m = re.match(r'(\d{6})-(\d+)', s, re.IGNORECASE)
    if m:
        info["date"] = m.group(1)
        info["plate"] = "p1"
        info["xy_raw"] = m.group(2)
        return info

    # Fallback
    m = re.search(r'(\d+)$', s)
    if m:
        info["xy_raw"] = m.group(1)
    return info


def folder_matches_date_plate(folder_name: str, date: str, plate: str) -> bool:
    """
    Does folder_name correspond to the given date and plate?
      date=240731, plate=p2  -> "240731_plate2*"
      date=240822, plate=p1  -> "20240822_cy_plate1"
      date=240725, plate=r1  -> "20240725_r1"
    """
    fn = folder_name.lower()
    date_long = "20" + date  # 240731 -> 20240731
    has_date = (date in fn) or (date_long in fn)
    if not has_date:
        return False
    if plate is None:
        return True

    plate = plate.lower()
    plate_as_word = re.sub(r'^p', 'plate', plate)   # p2 -> plate2
    return (plate_as_word in fn) or (plate in fn)


def tif_matches_xy(tif_name: str, xy_raw: str) -> bool:
    """
    Exact xy match: xy_raw="02" matches xy02 but NOT xy12, xy22, etc.

    Strategy: strip leading zeros to get the canonical integer (e.g. "2"),
    then require  'xy' + optional-zeros + that-integer + (non-digit or end),
    OR separator + optional-zeros + that-integer + end-of-stem.
    """
    fn = tif_name.lower()
    if fn.endswith(".tif"):
        fn = fn[:-4]

    num = xy_raw.lstrip("0") or "0"   # "02" -> "2",  "18" -> "18"

    # xy-prefixed: xy02, xy002, xy2 — all mean xy=2, must NOT be followed by digit
    if re.search(rf'xy0*{re.escape(num)}(?:\D|$)', fn):
        return True

    # Separator at end of stem: -02, _02, -002 etc.
    if re.search(rf'[-_]0*{re.escape(num)}$', fn):
        return True

    return False


# ── MAIN ─────────────────────────────────────────────────────────────────────

def collect_all_tifs():
    tif_files = []
    for subfolder in SUBFOLDERS:
        folder = DATADIR / subfolder
        if not folder.exists():
            continue
        for root, dirs, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(".tif"):
                    tif_files.append(Path(root) / f)
    # Loose tifs directly in DATADIR
    if DATADIR.exists():
        for f in DATADIR.iterdir():
            if f.is_file() and f.suffix.lower() == ".tif":
                tif_files.append(f)
    return tif_files


def find_matches(tif_files, parsed_id):
    date = parsed_id["date"]
    plate = parsed_id["plate"]
    xy_raw = parsed_id["xy_raw"]
    filename_prefix = parsed_id.get("filename_prefix")
    matches = []

    for tif in tif_files:
        fname = tif.name
        fname_lower = fname.lower()

        # Folder check (also try grandparent for nested structures)
        if date:
            folder_ok = folder_matches_date_plate(tif.parent.name, date, plate)
            if not folder_ok:
                folder_ok = folder_matches_date_plate(tif.parent.parent.name, date, plate)
            if not folder_ok:
                continue

        # Filename prefix check (e.g. "c1" for plate3c1 files)
        if filename_prefix and filename_prefix not in fname_lower:
            continue

        # Exact xy match
        if xy_raw and not tif_matches_xy(fname, xy_raw):
            continue

        matches.append(tif)

    return matches


def main():
    print(f"\nScanning: {DATADIR}\n")
    tif_files = collect_all_tifs()
    print(f"Found {len(tif_files)} .tif files total.\n")

    results = []
    for (reporter, condition), raw_ids in RAW_TABLE.items():
        for raw_id in raw_ids:
            parsed = parse_file_id(raw_id)
            matches = find_matches(tif_files, parsed)
            results.append({
                "reporter": reporter,
                "condition": condition,
                "file_id": raw_id,
                "parsed_date": parsed["date"] or "",
                "parsed_plate": parsed["plate"] or "",
                "parsed_prefix": parsed.get("filename_prefix") or "",
                "parsed_xy": parsed["xy_raw"] or "",
                "n_matches": len(matches),
                "matched_files": "; ".join(str(p) for p in matches) if matches else "NOT FOUND",
            })

    # Print table
    hdr = f"{'Reporter':<14} {'Condition':<14} {'FileID':<22} {'Date':<8} {'Plate':<6} {'Prefix':<8} {'XY':<5} {'N':<3} Matched Files"
    print(hdr)
    print("-" * 130)
    for r in results:
        print(
            f"{r['reporter']:<14} {r['condition']:<14} {r['file_id']:<22} "
            f"{r['parsed_date']:<8} {r['parsed_plate']:<6} {r['parsed_prefix']:<8} {r['parsed_xy']:<5} "
            f"{r['n_matches']:<3} {r['matched_files']}"
        )

    # Save CSV
    out_csv = Path("tif_file_mapping.csv")
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)
    print(f"\nResults saved to: {out_csv.resolve()}")

    missing = [r for r in results if r["n_matches"] == 0]
    if missing:
        print(f"\n⚠️  {len(missing)} entries NOT FOUND:")
        for r in missing:
            print(f"   {r['reporter']} / {r['condition']} / {r['file_id']}")
    else:
        print("\n✓ All entries matched.")


if __name__ == "__main__":
    main()


Scanning: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging

Found 1985 .tif files total.

Reporter       Condition      FileID                 Date     Plate  Prefix   XY    N   Matched Files
----------------------------------------------------------------------------------------------------------------------------------
tbk1           blank          240731p2-50            240731   p2              50    1   ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy50.tif
tbk1           polyIC         0731p2-57              240731   p2              57    1   ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy57.tif
tbk1           polyIC         0725r1xy18             240725   r1              18    1   ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20240725_r1/20240725_r1c1xy18.tif
tbk1           diABZI         0731p2-62              240731   p2              62    1   ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy62.tif
tbk1           BMS-345541

In [14]:
"""
find_tif_files_v2.py

Outputs a list of .tif filepaths for each (date, plate, reporter, condition)
combination parsed from the layout tables.

Usage:
    python find_tif_files_v2.py

Output:
    - tif_file_mapping_v2.csv  : one row per file
    - tif_file_mapping_v2.txt  : grouped, human-readable
"""

import os
import re
import csv
from pathlib import Path
from collections import defaultdict

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATADIR = Path("${SPARK_ACQUISITION_ROOT}/ExRai-imaging")

SUBFOLDERS = [
    "20240321-ECL_CY_HT1080_ExRai_polydIdC_2ug_ml_trials002",
    "20240321-ECL_CY_HT1080_ExRai_polyIC_2ug_ml_trials002",
    "20240404-CY_HT1080_ExRai_1uM_diABZI_",
    "20240404-_CY_HT1080_ExRai_polyIC_7ug_ml",
    "20240516-_CY_HT1080_ExRai_abiZI_001",
    "20240516-_cy_ht1080_exrai_polyic",
    "20240516-_CY_HT1080_ExRai_polyIC",
    "20240606-_CY_HT1080_ExRai_diABZI001",
    "20240606-_CY_HT1080_ExRai_polyIC002",
    "20240627-plate1-polyIC_600", "20240627-plate2-polyIC_600",
    "20240628-plate1-polyIC_600", "20240628-plate2-polyIC_600",
    "20240718-CY_spk_diabzi_1uM_2uM",
    "20240719-CY_A3_fast", "20240719-CY_A3_fast002", "20240719-CY_all",
    "20240725_F5", "20240725_F6", "20240725_F7",
    "20240725_F8", "20240725_F9",
    "20240725_r1", "20240725_r2",
    "20240814_cy_plate1", "20240814_cy_plate2",
    "20240815_cy_plate3", "20240815_cy_plate4",
    "20240822_cy_plate1", "20240822_cy_plate2",
    "20250126-plate1", "20250130-plate1",
    "240731_plate1", "240731_plate1_final",
    "240731_plate2", "240731_plate3",
    "240801_plate4",
]

# ── TABLE DEFINITIONS ────────────────────────────────────────────────────────
# Each entry: (date, plate, reporter, condition, [xy numbers as strings])
# xy numbers are kept as original strings to preserve zero-padding info.
# Row letter (C/D/E/F) is recorded but not used for file matching —
# the xy numbers already uniquely identify files within the folder.

ENTRIES = [
    # ── 240731 plate2 ─────────────────────────────────────────────────────
    # Columns: blank | blank+GSK8612_24h | polyIC | diABZI
    ("240731", "plate2", "IKK-SPK-1",   "blank",           ["1","2"]),
    ("240731", "plate2", "IKK-SPK-1",   "blank+GSK8612",   ["4","6"]),
    ("240731", "plate2", "IKK-SPK-1",   "polyIC",          ["8","9"]),
    ("240731", "plate2", "IKK-SPK-1",   "diABZI",          ["11","13"]),

    ("240731", "plate2", "TBK-SPK-1",   "blank",           ["15","16"]),
    ("240731", "plate2", "TBK-SPK-1",   "blank+GSK8612",   ["18","20"]),
    ("240731", "plate2", "TBK-SPK-1",   "polyIC",          ["22","25"]),
    ("240731", "plate2", "TBK-SPK-1",   "diABZI",          ["26","27"]),

    ("240731", "plate2", "IKK-SPK-1",   "blank",           ["30","32"]),
    ("240731", "plate2", "IKK-SPK-1",   "blank+GSK8612",   ["33","36"]),
    ("240731", "plate2", "IKK-SPK-1",   "polyIC",          ["37","39","40","41"]),
    ("240731", "plate2", "IKK-SPK-1",   "diABZI",          ["44","46"]),

    ("240731", "plate2", "TBK-SPK-1",   "blank",           ["49","50"]),
    ("240731", "plate2", "TBK-SPK-1",   "blank+GSK8612",   ["52","54"]),
    ("240731", "plate2", "TBK-SPK-1",   "polyIC",          ["57","58"]),
    ("240731", "plate2", "TBK-SPK-1",   "diABZI",          ["62","64"]),

    # ── 240731 plate3 ─────────────────────────────────────────────────────
    # Columns: blank | GSK8612 | polyIC | diABZI
    ("240731", "plate3", "TBK-SPK-novel", "blank",          ["34","35"]),
    ("240731", "plate3", "TBK-SPK-novel", "GSK8612",        ["36","38"]),
    ("240731", "plate3", "TBK-SPK-novel", "polyIC",         ["40","44"]),
    ("240731", "plate3", "TBK-SPK-novel", "diABZI",         ["46","49"]),

    ("240731", "plate3", "TBK-SPK-novel", "blank",          ["50"]),
    ("240731", "plate3", "TBK-SPK-novel", "GSK8612",        ["54"]),
    ("240731", "plate3", "TBK-SPK-novel", "polyIC",         ["60"]),
    ("240731", "plate3", "TBK-SPK-novel", "diABZI",         ["62","64","65"]),

    # ── 240815 plate3 ─────────────────────────────────────────────────────
    # Columns: blank | GSK-8612 | TNFa | diABZI
    ("240815", "plate3", "TBK-SPK-1-TA",  "blank",          ["2"]),
    ("240815", "plate3", "TBK-SPK-1-TA",  "GSK-8612",       []),
    ("240815", "plate3", "TBK-SPK-1-TA",  "TNFa",           ["7","8"]),
    ("240815", "plate3", "TBK-SPK-1-TA",  "diABZI",         ["14","16"]),

    ("240815", "plate3", "TBK-SPK-1-TA",  "blank",          ["17"]),
    ("240815", "plate3", "TBK-SPK-1-TA",  "GSK-8612",       []),
    ("240815", "plate3", "TBK-SPK-1-TA",  "TNFa",           ["24","25"]),
    ("240815", "plate3", "TBK-SPK-1-TA",  "diABZI",         ["30","32"]),

    ("240815", "plate3", "IKK-SPK-1-TA",  "blank",          ["33","35"]),
    ("240815", "plate3", "IKK-SPK-1-TA",  "GSK-8612",       []),
    ("240815", "plate3", "IKK-SPK-1-TA",  "TNFa",           ["42","43"]),
    ("240815", "plate3", "IKK-SPK-1-TA",  "diABZI",         ["44"]),

    ("240815", "plate3", "IKK-SPK-1-TA",  "blank",          []),
    ("240815", "plate3", "IKK-SPK-1-TA",  "GSK-8612",       []),
    ("240815", "plate3", "IKK-SPK-1-TA",  "TNFa",           ["56","59"]),
    ("240815", "plate3", "IKK-SPK-1-TA",  "diABZI",         ["63","64"]),

    # ── 240815 plate4 ─────────────────────────────────────────────────────
    # Columns: blank | GSK-8612 | diABZI | polyIC | TNFa
    ("240815", "plate4", "IKK-SPK-novel", "blank",           ["29","30"]),
    ("240815", "plate4", "IKK-SPK-novel", "GSK-8612",        ["33","34"]),
    ("240815", "plate4", "IKK-SPK-novel", "diABZI",          []),
    ("240815", "plate4", "IKK-SPK-novel", "polyIC",          ["42"]),
    ("240815", "plate4", "IKK-SPK-novel", "TNFa",            []),

    ("240815", "plate4", "IKK-SPK-novel", "blank",           []),
    ("240815", "plate4", "IKK-SPK-novel", "GSK-8612",        []),
    ("240815", "plate4", "IKK-SPK-novel", "diABZI",          ["53","54"]),
    ("240815", "plate4", "IKK-SPK-novel", "polyIC",          ["55","56","58"]),
    ("240815", "plate4", "IKK-SPK-novel", "TNFa",            ["29","63","64"]),

    # ── 240822 plate2 ─────────────────────────────────────────────────────
    # Columns: blank | GSK-8612 | BMS-345541 | TNFa | polyIC
    ("240822", "plate2", "IKK-SPK-1",    "blank",            ["2","3"]),
    ("240822", "plate2", "IKK-SPK-1",    "GSK-8612",         ["4","5"]),
    ("240822", "plate2", "IKK-SPK-1",    "BMS-345541",       ["7","8","9"]),
    ("240822", "plate2", "IKK-SPK-1",    "TNFa",             ["10","13"]),
    ("240822", "plate2", "IKK-SPK-1",    "polyIC",           ["14","16"]),

    ("240822", "plate2", "IKK-SPK-novel","blank",            ["36","37"]),
    ("240822", "plate2", "IKK-SPK-novel","GSK-8612",         ["38","39"]),
    ("240822", "plate2", "IKK-SPK-novel","BMS-345541",       ["41"]),
    ("240822", "plate2", "IKK-SPK-novel","TNFa",             ["47"]),
    ("240822", "plate2", "IKK-SPK-novel","polyIC",           ["48","50","51","52"]),

    ("240822", "plate2", "TBK-SPK-novel","blank",            ["54","55"]),
    ("240822", "plate2", "TBK-SPK-novel","GSK-8612",         ["57","58"]),
    ("240822", "plate2", "TBK-SPK-novel","BMS-345541",       ["60","61"]),
    ("240822", "plate2", "TBK-SPK-novel","TNFa",             []),
    ("240822", "plate2", "TBK-SPK-novel","polyIC",           ["68","70","71"]),
]


# ── FILE MATCHING ─────────────────────────────────────────────────────────────

def collect_all_tifs():
    tif_files = []
    for subfolder in SUBFOLDERS:
        folder = DATADIR / subfolder
        if not folder.exists():
            continue
        for root, dirs, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(".tif"):
                    tif_files.append(Path(root) / f)
    if DATADIR.exists():
        for f in DATADIR.iterdir():
            if f.is_file() and f.suffix.lower() == ".tif":
                tif_files.append(f)
    return tif_files


def folder_matches(folder_name: str, date: str, plate: str) -> bool:
    """
    Match folder to date+plate.
      date=240731, plate=plate2 -> "240731_plate2*"
      date=240822, plate=plate2 -> "20240822_cy_plate2"
      date=240815, plate=plate3 -> "20240815_cy_plate3"
    """
    fn = folder_name.lower()
    date_long = "20" + date          # 240731 -> 20240731
    has_date = (date in fn) or (date_long in fn)
    if not has_date:
        return False
    plate_lc = plate.lower()         # "plate2"
    return plate_lc in fn


def tif_matches_xy(tif_name: str, xy_raw: str) -> bool:
    """
    Exact xy match. xy_raw e.g. "02", "57".
    Strips leading zeros to get canonical integer, then matches
    xy0*<num>(non-digit|end) or separator+0*<num>+end.
    """
    fn = tif_name.lower()
    if fn.endswith(".tif"):
        fn = fn[:-4]
    num = xy_raw.lstrip("0") or "0"
    if re.search(rf'xy0*{re.escape(num)}(?:\D|$)', fn):
        return True
    if re.search(rf'[-_]0*{re.escape(num)}$', fn):
        return True
    return False


def find_file(tif_files, date, plate, xy_raw):
    """Return matching tif path(s) for a single xy number."""
    matches = []
    for tif in tif_files:
        folder = tif.parent.name
        # check folder (also grandparent for nested)
        if not folder_matches(folder, date, plate):
            if not folder_matches(tif.parent.parent.name, date, plate):
                continue
        if tif_matches_xy(tif.name, xy_raw):
            matches.append(tif)
    return matches


# ── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print(f"\nScanning: {DATADIR}\n")
    tif_files = collect_all_tifs()
    print(f"Found {len(tif_files)} .tif files total.\n")

    # Merge entries with same (date, plate, reporter, condition)
    merged = defaultdict(list)
    for date, plate, reporter, condition, xys in ENTRIES:
        key = (date, plate, reporter, condition)
        for xy in xys:
            if xy not in merged[key]:
                merged[key].append(xy)

    csv_rows = []
    txt_lines = []
    not_found = []

    for (date, plate, reporter, condition), xys in merged.items():
        group_header = f"[{date} {plate}] {reporter} | {condition}"
        txt_lines.append(group_header)

        for xy in sorted(xys, key=lambda x: int(x)):
            matches = find_file(tif_files, date, plate, xy)
            if not matches:
                path_str = "NOT FOUND"
                not_found.append((date, plate, reporter, condition, xy))
            else:
                path_str = str(matches[0])
                if len(matches) > 1:
                    path_str += f"  [WARNING: {len(matches)} matches]"

            txt_lines.append(f"  xy{xy.zfill(2)}: {path_str}")
            csv_rows.append({
                "date": date,
                "plate": plate,
                "reporter": reporter,
                "condition": condition,
                "xy": xy,
                "n_matches": len(matches),
                "filepath": str(matches[0]) if matches else "NOT FOUND",
            })

        txt_lines.append("")

    # Print
    for line in txt_lines:
        print(line)

    # Save CSV
    out_csv = Path("tif_file_mapping_v2.csv")
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["date","plate","reporter","condition","xy","n_matches","filepath"])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"CSV saved to: {out_csv.resolve()}")

    # Save TXT
    out_txt = Path("tif_file_mapping_v2.txt")
    with open(out_txt, "w") as f:
        f.write("\n".join(txt_lines))
    print(f"TXT saved to: {out_txt.resolve()}")

    if not_found:
        print(f"\n⚠️  {len(not_found)} xy positions NOT FOUND:")
        for date, plate, reporter, condition, xy in not_found:
            print(f"   {date} {plate} | {reporter} | {condition} | xy{xy.zfill(2)}")
    else:
        print("\n✓ All xy positions matched.")


In [15]:
main()


Scanning: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging

Found 1985 .tif files total.

[240731 plate2] IKK-SPK-1 | blank
  xy01: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy01.tif
  xy02: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy02.tif
  xy30: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy30.tif
  xy32: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy32.tif

[240731 plate2] IKK-SPK-1 | blank+GSK8612
  xy04: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy04.tif
  xy06: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy06.tif
  xy33: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy33.tif
  xy36: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy36.tif

[240731 plate2] IKK-SPK-1 | polyIC
  xy08: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/240731_plate2/240731_plate2c1xy08.tif
  xy09: ${SPARK_ACQUISITION_ROO

In [3]:
"""
find_tif_files_v3.py

Outputs .tif filepaths for each (reporter, condition) from the 20250126 plate1 table.

Table layout (Apo channels 4-9):
             blank    diABZI(6)  diABZI(7)  polyIC   GSK-8612  BMS-345541
             col4     col5       col6       col7     col8      col9
TBK-SPK-1    3,5      11         14,15      19       21,22     23
TBK-SPK-1-TA 24       25         26         27       28        29
TBK-SPK-novel 32,33,34 37,38,39,41 44       49,54    55,56     60
TBK-SPK-novel-TA  -   61         62         63       64        -

Both diABZI columns (Apo5 and Apo6) are merged into a single "diABZI" condition.

Usage:
    python find_tif_files_v3.py

Output:
    - tif_file_mapping_v3.csv
    - tif_file_mapping_v3.txt
"""

import os
import re
import csv
from pathlib import Path
from collections import defaultdict

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATADIR = Path("${SPARK_ACQUISITION_ROOT}/ExRai-imaging")
FOLDER  = "20250126-plate1"

# ── TABLE ────────────────────────────────────────────────────────────────────
# (reporter, condition, [xy numbers as strings])
# Both diABZI columns (Apo5 and Apo6) merged into a single "diABZI" condition
ENTRIES = [
    ("TBK-SPK-1",        "blank",      ["3", "5"]),
    ("TBK-SPK-1",        "diABZI",     ["11"]),
    ("TBK-SPK-1",        "diABZI",     ["14", "15"]),
    ("TBK-SPK-1",        "polyIC",     ["19"]),
    ("TBK-SPK-1",        "GSK-8612",   ["21", "22"]),
    ("TBK-SPK-1",        "BMS-345541", ["23"]),

    ("TBK-SPK-1-TA",     "blank",      ["24"]),
    ("TBK-SPK-1-TA",     "diABZI",     ["25"]),
    ("TBK-SPK-1-TA",     "diABZI",     ["26"]),
    ("TBK-SPK-1-TA",     "polyIC",     ["27"]),
    ("TBK-SPK-1-TA",     "GSK-8612",   ["28"]),
    ("TBK-SPK-1-TA",     "BMS-345541", ["29"]),

    ("TBK-SPK-novel",    "blank",      ["32", "33", "34"]),
    ("TBK-SPK-novel",    "diABZI",     ["37", "38", "39", "41"]),
    ("TBK-SPK-novel",    "diABZI",     ["44"]),
    ("TBK-SPK-novel",    "polyIC",     ["49", "54"]),
    ("TBK-SPK-novel",    "GSK-8612",   ["55", "56"]),
    ("TBK-SPK-novel",    "BMS-345541", ["60"]),

    ("TBK-SPK-novel-TA", "blank",      []),
    ("TBK-SPK-novel-TA", "diABZI",     ["61"]),
    ("TBK-SPK-novel-TA", "diABZI",     ["62"]),
    ("TBK-SPK-novel-TA", "polyIC",     ["63"]),
    ("TBK-SPK-novel-TA", "GSK-8612",   ["64"]),
    ("TBK-SPK-novel-TA", "BMS-345541", []),
]


# ── FILE MATCHING ─────────────────────────────────────────────────────────────

def collect_tifs_in_folder(folder_path: Path):
    tif_files = []
    if not folder_path.exists():
        print(f"⚠️  Folder not found: {folder_path}")
        return tif_files
    for root, dirs, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".tif"):
                tif_files.append(Path(root) / f)
    return tif_files


def tif_matches_xy(tif_name: str, xy_raw: str) -> bool:
    """
    Exact xy match. xy_raw e.g. "3", "11", "54".
    Matches xy0*<num>(non-digit|end) or separator+0*<num>+end.
    """
    fn = tif_name.lower()
    if fn.endswith(".tif"):
        fn = fn[:-4]
    num = xy_raw.lstrip("0") or "0"
    if re.search(rf'xy0*{re.escape(num)}(?:\D|$)', fn):
        return True
    if re.search(rf'[-_]0*{re.escape(num)}$', fn):
        return True
    return False


def find_file(tif_files, xy_raw):
    return [t for t in tif_files if tif_matches_xy(t.name, xy_raw)]


# ── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    folder_path = DATADIR / FOLDER
    print(f"\nScanning: {folder_path}\n")
    tif_files = collect_tifs_in_folder(folder_path)
    print(f"Found {len(tif_files)} .tif files.\n")

    # Merge duplicate (reporter, condition) keys
    merged = defaultdict(list)
    for reporter, condition, xys in ENTRIES:
        key = (reporter, condition)
        for xy in xys:
            if xy not in merged[key]:
                merged[key].append(xy)

    csv_rows = []
    txt_lines = []
    not_found = []

    for (reporter, condition), xys in merged.items():
        group_header = f"[20250126 plate1] {reporter} | {condition}"
        txt_lines.append(group_header)

        if not xys:
            txt_lines.append("  (no entries)")
            txt_lines.append("")
            continue

        for xy in sorted(xys, key=lambda x: int(x)):
            matches = find_file(tif_files, xy)
            if not matches:
                path_str = "NOT FOUND"
                not_found.append((reporter, condition, xy))
            else:
                path_str = str(matches[0])
                if len(matches) > 1:
                    path_str += f"  [WARNING: {len(matches)} matches — {', '.join(str(m) for m in matches[1:])}]"

            txt_lines.append(f"  xy{xy.zfill(2)}: {path_str}")
            csv_rows.append({
                "date": "20250126",
                "plate": "plate1",
                "reporter": reporter,
                "condition": condition,
                "xy": xy,
                "n_matches": len(matches),
                "filepath": str(matches[0]) if matches else "NOT FOUND",
            })

        txt_lines.append("")

    # Print
    for line in txt_lines:
        print(line)

    # Save CSV
    out_csv = Path("tif_file_mapping_v3.csv")
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["date","plate","reporter","condition","xy","n_matches","filepath"])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"CSV saved to: {out_csv.resolve()}")

    # Save TXT
    out_txt = Path("tif_file_mapping_v3.txt")
    with open(out_txt, "w") as f:
        f.write("\n".join(txt_lines))
    print(f"TXT saved to: {out_txt.resolve()}")

    if not_found:
        print(f"\n⚠️  {len(not_found)} xy positions NOT FOUND:")
        for reporter, condition, xy in not_found:
            print(f"   {reporter} | {condition} | xy{xy.zfill(2)}")
    else:
        print("\n✓ All xy positions matched.")





In [5]:
main()


Scanning: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1

Found 61 .tif files.

[20250126 plate1] TBK-SPK-1 | blank
  xy03: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy03.tif
  xy05: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy05.tif

[20250126 plate1] TBK-SPK-1 | diABZI
  xy11: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy11.tif
  xy14: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy14.tif
  xy15: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy15.tif

[20250126 plate1] TBK-SPK-1 | polyIC
  xy19: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy19.tif

[20250126 plate1] TBK-SPK-1 | GSK-8612
  xy21: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy21.tif
  xy22: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250126-plate1/20250126-plate1001c1xy22.tif

[20250126 pla

In [8]:
"""
find_tif_files_v4.py

Outputs .tif filepaths for each (reporter, condition) from the 20250130 plate1 table.

Table layout (Apo channels 4-9):
              blank    GSK-8612  diABZI    LPS       diABZI    LPS
              Apo4     Apo5      Apo6      Apo7      Apo8      Apo9
TBK-SPK-1     4,5      7         13        16,18     20        26,27,28
TBK-SPK-1-TA  30,32    34,35,36  39,40,43,44  45     50,53     55,57,58
TBK-SPK-novel 60,61    62        /         /         /         104
TBK-SPK-novel-TA 79,80,81 /     85,89     /         92,93,96  98,99

Both diABZI columns (Apo6 and Apo8) are merged into a single "diABZI" condition.
Both LPS columns (Apo7 and Apo9) are merged into a single "LPS" condition.
/ means no FOV selected (empty).

Usage:
    python find_tif_files_v4.py

Output:
    - tif_file_mapping_v4.csv
    - tif_file_mapping_v4.txt
"""

import os
import re
import csv
from pathlib import Path
from collections import defaultdict

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATADIR = Path("${SPARK_ACQUISITION_ROOT}/ExRai-imaging")
FOLDER  = "20250130-plate1"

# ── TABLE ────────────────────────────────────────────────────────────────────
# (reporter, condition, [xy numbers as strings])
# Duplicate condition columns (diABZI x2, LPS x2) are merged per reporter row.
ENTRIES = [
    # Columns: blank | GSK-8612 | diABZI | LPS | diABZI | LPS
    ("TBK-SPK-1",        "blank",    ["4", "5"]),
    ("TBK-SPK-1",        "GSK-8612", ["7"]),
    ("TBK-SPK-1",        "diABZI",   ["13", "20"]),
    ("TBK-SPK-1",        "LPS",      ["16", "18", "26", "27", "28"]),

    ("TBK-SPK-1-TA",     "blank",    ["30", "32"]),
    ("TBK-SPK-1-TA",     "GSK-8612", ["34", "35", "36"]),
    ("TBK-SPK-1-TA",     "diABZI",   ["39", "40", "43", "44", "50", "53"]),
    ("TBK-SPK-1-TA",     "LPS",      ["45", "55", "57", "58"]),

    ("TBK-SPK-novel",    "blank",    ["60", "61"]),
    ("TBK-SPK-novel",    "GSK-8612", ["62"]),
    ("TBK-SPK-novel",    "LPS",      ["104"]),

    ("TBK-SPK-novel-TA", "blank",    ["79", "80", "81"]),
    ("TBK-SPK-novel-TA", "diABZI",   ["85", "89", "92", "93", "96"]),
    ("TBK-SPK-novel-TA", "LPS",      ["98", "99"]),
]


# ── FILE MATCHING ─────────────────────────────────────────────────────────────

def collect_tifs_in_folder(folder_path: Path):
    tif_files = []
    if not folder_path.exists():
        print(f"⚠️  Folder not found: {folder_path}")
        return tif_files
    for root, dirs, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".tif"):
                tif_files.append(Path(root) / f)
    return tif_files


def tif_matches_xy(tif_name: str, xy_raw: str) -> bool:
    """
    Exact xy match. xy_raw e.g. "04", "13", "104".
    Strips leading zeros to get canonical integer, then matches
    xy0*<num>(non-digit|end) or separator+0*<num>+end.
    """
    fn = tif_name.lower()
    if fn.endswith(".tif"):
        fn = fn[:-4]
    num = xy_raw.lstrip("0") or "0"
    if re.search(rf'xy0*{re.escape(num)}(?:\D|$)', fn):
        return True
    if re.search(rf'[-_]0*{re.escape(num)}$', fn):
        return True
    return False


def find_file(tif_files, xy_raw):
    return [t for t in tif_files if tif_matches_xy(t.name, xy_raw)]


# ── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    folder_path = DATADIR / FOLDER
    print(f"\nScanning: {folder_path}\n")
    tif_files = collect_tifs_in_folder(folder_path)
    print(f"Found {len(tif_files)} .tif files.\n")

    # Merge duplicate (reporter, condition) keys
    merged = defaultdict(list)
    for reporter, condition, xys in ENTRIES:
        key = (reporter, condition)
        for xy in xys:
            if xy not in merged[key]:
                merged[key].append(xy)

    csv_rows = []
    txt_lines = []
    not_found = []

    for (reporter, condition), xys in merged.items():
        group_header = f"[20250130 plate1] {reporter} | {condition}"
        txt_lines.append(group_header)

        if not xys:
            txt_lines.append("  (no entries)")
            txt_lines.append("")
            continue

        for xy in sorted(xys, key=lambda x: int(x)):
            matches = find_file(tif_files, xy)
            if not matches:
                path_str = "NOT FOUND"
                not_found.append((reporter, condition, xy))
            else:
                path_str = str(matches[0])
                if len(matches) > 1:
                    path_str += f"  [WARNING: {len(matches)} matches — {', '.join(str(m) for m in matches[1:])}]"

            txt_lines.append(f"  xy{xy.zfill(2)}: {path_str}")
            csv_rows.append({
                "date": "20250130",
                "plate": "plate1",
                "reporter": reporter,
                "condition": condition,
                "xy": xy,
                "n_matches": len(matches),
                "filepath": str(matches[0]) if matches else "NOT FOUND",
            })

        txt_lines.append("")

    # Print
    for line in txt_lines:
        print(line)

    # Save CSV
    out_csv = Path("tif_file_mapping_v4.csv")
    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["date","plate","reporter","condition","xy","n_matches","filepath"])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"CSV saved to: {out_csv.resolve()}")

    # Save TXT
    out_txt = Path("tif_file_mapping_v4.txt")
    with open(out_txt, "w") as f:
        f.write("\n".join(txt_lines))
    print(f"TXT saved to: {out_txt.resolve()}")

    if not_found:
        print(f"\n⚠️  {len(not_found)} xy positions NOT FOUND:")
        for reporter, condition, xy in not_found:
            print(f"   {reporter} | {condition} | xy{xy.zfill(2)}")
    else:
        print("\n✓ All xy positions matched.")



In [9]:
main()


Scanning: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1

Found 105 .tif files.

[20250130 plate1] TBK-SPK-1 | blank
  xy04: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy004c1.tif
  xy05: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy005c1.tif

[20250130 plate1] TBK-SPK-1 | GSK-8612
  xy07: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy007c1.tif

[20250130 plate1] TBK-SPK-1 | diABZI
  xy13: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy013c1.tif
  xy20: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy020c1.tif

[20250130 plate1] TBK-SPK-1 | LPS
  xy16: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy016c1.tif
  xy18: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy018c1.tif
  xy26: ${SPARK_ACQUISITION_ROOT}/ExRai-imaging/20250130-plate1/20250130-plate1xy026c1.tif
  xy27: ${SPARK_ACQUISITION_ROOT